# Notebook 17 · Ensayos complementarios para revisión editorial
**Manuscrito:** *Geometric Invariance and Representation Sufficiency in Morphometric Bean and Rice Classification* · **AgriEngineering — 4664292**  
**Estado:** análisis suplementarios exploratorios derivados de la revisión, no nuevas hipótesis preespecificadas.  
**Hardware:** **CPU**. No se requiere GPU, TPU ni entrenamiento de modelos.  

### Objetivos
1. Relacionar el deterioro por clase en **Dry Bean** con el perfil de **área en píxeles**; enfoque especial en SIRA y DERMASON.
2. Analizar la **sensibilidad de los márgenes** de equivalencia (±0,005, ±0,01 y ±0,02), IC pareados agrupados y precisión estadística retrospectiva (NO una justificación agronómica).
3. Comparar **APS aleatorizado, LAC, RAPS y Mondrian LAC** usando datos de calibración independientes del ajuste de temperatura y umbral selectivo.
4. Producir un **ZIP único NB17_ENSAYOS_COMPLEMENTARIOS_4664292.zip**, con tablas, figuras y hashes SHA-256 para revisión y GitHub.

**Principio de reproducibilidad:** el conjunto test congelado nunca se usa para selección de modelos, hiperparámetros, métodos, penalizaciones o cuantiles. RAPS utiliza parámetros fijados a priori y no optimizados con el test. Los resultados de conformal no garantizan cobertura condicional individual ni generalización a otras campañas o lotes.

## 0. Instalación y configuración
Ejecuta **Entorno de ejecución → Cambiar tipo de entorno → CPU**, después **Ejecutar todas**. El notebook buscará el ZIP del Notebook 12 en tu Drive. Para el perfil morfométrico descargará el CSV congelado del repositorio científico, verificará su SHA-256 y no lo modificará.

In [ ]:
import sys, subprocess, importlib.util
needed={'pyarrow':'pyarrow','scipy':'scipy','scikit-learn':'sklearn','statsmodels':'statsmodels','matplotlib':'matplotlib','pandas':'pandas'}
missing=[pip for pip,mod in needed.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from pathlib import Path
import os,json,hashlib,zipfile,shutil,platform,urllib.request,time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, log_loss, confusion_matrix
from IPython.display import display
SEED=20261027
N_BOOT=2000
ALPHAS=(0.10,0.05)
MARGINS=(0.005,0.010,0.020)
RAPS_LAMBDA=0.010    # sensiblidad predefinida: NO seleccionada usando el test
RAPS_KREG=1         # rango a partir del cual se penaliza la longitud
EXPECTED_DRY_SHA='e8938d0ffde0c65f73817d5797030a8335939aa330b8ce291fd59096f833165e'
print('CPU | Python',sys.version.split()[0],'| remuestreos',N_BOOT)


## 1. Google Drive y fuentes congeladas
Se utilizan las predicciones de **Notebook 12**, el ZIP previamente validado del **Notebook 16** y la tabla original congelada del repositorio. No se reentrena ni se cambia ninguna partición. Todos los archivos nuevos se guardan aparte.

In [ ]:
from google.colab import drive
if not (Path('/content/drive/MyDrive').is_dir()): drive.mount('/content/drive')
MYDRIVE=Path('/content/drive/MyDrive')
PROJECT=MYDRIVE/'Grain_Robustness_Q1'
REVIEW=PROJECT/'Revision_4664292_Ensayos_2026'
if not PROJECT.is_dir(): raise FileNotFoundError('No encuentro Grain_Robustness_Q1 en la cuenta de Drive conectada.')
REVIEW.mkdir(parents=True,exist_ok=True)
OUT=REVIEW/'resultados_colab'/'NB17_4664292'
OUT.mkdir(parents=True,exist_ok=True)
SRC_NAME='NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip'
choices=list(PROJECT.rglob(SRC_NAME))
if len(choices)!=1: raise FileNotFoundError(f'Se esperaba un solo {SRC_NAME}, se encontraron: {choices}')
NB12_ZIP=choices[0]
NB16_ZIP=REVIEW/'NB16_RESULTADOS_REVISORES_4664292.zip'
if not NB16_ZIP.exists():
    # NB16 no se usa para sustituir las predicciones; sólo como auditoría de comparación.
    raise FileNotFoundError(f'No está {NB16_ZIP}. Guarda allí el ZIP validado de NB16.')
WORK=Path('/content/nb17_src')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(NB12_ZIP) as z:
    assert z.testzip() is None,'ZIP NB12 dañado'
    z.extractall(WORK)
with zipfile.ZipFile(NB16_ZIP) as z:
    assert z.testzip() is None,'ZIP NB16 dañado'
    ctrl=pd.read_csv(z.open('NB16_03_McNemar_agrupado_y_bootstrap.csv'))
print('NB12:',NB12_ZIP.name, 'SHA256:',hashlib.sha256(NB12_ZIP.read_bytes()).hexdigest())
print('NB16:',NB16_ZIP.name, 'SHA256:',hashlib.sha256(NB16_ZIP.read_bytes()).hexdigest())
print('Guardar resultados NB17 en:',OUT)


## 2. Predicciones confirmatorias y control de alineación
Se validan las etiquetas, `record_id`, `duplicate_group_id` y métricas exactas publicadas en el repositorio antes de producir cualquier análisis adicional.

In [ ]:
REPS={
 'dry_bean': {'R1':'R1_full_native','R3-C':'R3_complete_invariant','R4-C':'R4_complete_log_area'},
 'iniap_rice':{'R1':'R1_full_native','R3-NR':'R3_complete_invariant','R4-NR':'R4_complete_log_area'}
}
EXPECT={('dry_bean','R1'):0.935439598153466,('dry_bean','R3-C'):0.7620670285392909,
 ('dry_bean','R4-C'):0.9330283682823068,('iniap_rice','R1'):0.977586916037337,
 ('iniap_rice','R3-NR'):0.97814222501307,('iniap_rice','R4-NR'):0.9811650961412388}
def load_pred(dataset,rep,part):
    internal=REPS[dataset][rep]
    matches=[p for p in WORK.rglob(f'rbf_svm_{part}_predictions.*')
          if p.parent.name==internal and p.parent.parent.name==dataset and p.suffix in ('.parquet','.csv')]
    matches.sort(key=lambda p:0 if p.suffix=='.parquet' else 1)
    if not matches: raise FileNotFoundError((dataset,rep,part))
    p=matches[0]
    frame=pd.read_parquet(p) if p.suffix=='.parquet' else pd.read_csv(p)
    for key in ('record_id','class_label','duplicate_group_id'):
        if key not in frame: raise AssertionError('Falta '+key)
        frame[key]=frame[key].astype(str)
    if frame.record_id.duplicated().any(): raise AssertionError('IDs duplicados')
    frame=frame.sort_values('record_id').reset_index(drop=True)
    return frame
def probs(frame):
    cols=sorted([c for c in frame if c.startswith('prob_native_')],key=lambda c:int(c.rsplit('_',1)[1]))
    if not cols: raise KeyError('No hay prob_native_ en predicciones NB12')
    classes=np.array([str(frame[f'class_{int(c.rsplit("_",1)[1])}'].iloc[0]) for c in cols])
    for j in range(len(classes)):
        assert frame[f'class_{j}'].astype(str).nunique()==1
    P=frame[cols].to_numpy(float)
    assert np.allclose(P.sum(1),1,atol=1e-5) and np.isfinite(P).all()
    return P,classes
PRED={}; audit=[]
for dataset,repmap in REPS.items():
    ref=None
    for name in repmap:
        f=load_pred(dataset,name,'test')
        keys=f[['record_id','class_label','duplicate_group_id']]
        if ref is None:ref=keys
        else: assert keys.equals(ref),'La alineación confirmatoria no coincide'
        p,c=probs(f)
        yh=c[p.argmax(1)]
        score=f1_score(f.class_label,yh,average='macro')
        if not np.isclose(score,EXPECT[(dataset,name)],atol=5e-5):
            raise AssertionError(f'{dataset}/{name}: {score} distinto del NB12')
        PRED[(dataset,name)]=f
        audit.append(dict(dataset=dataset,representation=name,macro_f1=score,n_test=len(f),n_groups=f.duplicate_group_id.nunique()))
audit_df=pd.DataFrame(audit)
audit_df.to_csv(OUT/'NB17_00_control_predicciones_confirmatorias.csv',index=False)
display(audit_df)


## 3. Perfiles de área y errores SIRA ↔ DERMASON
Usamos la tabla congelada de 13.611 granos de Dry Bean, sin cambiar los datos ni repartir de nuevo los grupos. Se comparan distribuciones del área (en **píxeles²**) y errores del mismo test confirmatorio. **La asociación entre área y errores no prueba causalidad**.

In [ ]:
SOURCE_RAW='https://raw.githubusercontent.com/Julians30/geometric-invariance-bean-rice/main/data/dry_bean_frozen_v1.csv'
DRY_CSV=REVIEW/'dry_bean_frozen_v1.csv'
if not DRY_CSV.exists():
    try:
        urllib.request.urlretrieve(SOURCE_RAW,DRY_CSV)
    except Exception as ex:
        print('GitHub no respondió:',str(ex)[:220])
        print('Descarga manualmente data/dry_bean_frozen_v1.csv del repositorio de GitHub y súbelo aquí.')
        from google.colab import files
        up=files.upload()
        if 'dry_bean_frozen_v1.csv' not in up: raise FileNotFoundError('Falta dry_bean_frozen_v1.csv')
        DRY_CSV.write_bytes(up['dry_bean_frozen_v1.csv'])
sha=hashlib.sha256(DRY_CSV.read_bytes()).hexdigest()
if sha!=EXPECTED_DRY_SHA:
    raise AssertionError('El Dry Bean descargado no coincide con el SHA-256 congelado. NO seguir.')
dry=pd.read_csv(DRY_CSV)
required={'record_id','Area','Class','duplicate_group_id'}
if not required.issubset(dry.columns): raise KeyError(f'Faltan {required-set(dry.columns)}')
assert len(dry)==13611 and dry.record_id.is_unique
assert (dry.Area>0).all()
print('Dry Bean congelado: verificado',len(dry),'registros; SHA256:',sha)
# Perfiles descriptivos por clase — conjunto completo (sin selección a posteriori)
profiles=dry.groupby('Class').agg(n=('Area','size'),area_mean=('Area','mean'),area_sd=('Area','std'),
  area_median=('Area','median'),area_q10=('Area',lambda x:x.quantile(.10)),
  area_q25=('Area',lambda x:x.quantile(.25)),area_q75=('Area',lambda x:x.quantile(.75)),
  area_q90=('Area',lambda x:x.quantile(.90))).reset_index().sort_values('Class')
profiles.to_csv(OUT/'NB17_01_perfiles_area_por_clase_dry_bean.csv',index=False)
display(profiles.round(1))
fig,ax=plt.subplots(figsize=(11,5))
labels=sorted(dry.Class.unique())
ax.boxplot([dry.loc[dry.Class==cl,'Area'].to_numpy() for cl in labels],tick_labels=labels,showfliers=False)
ax.set_ylabel('Área (píxeles²)');ax.set_title('Dry Bean — distribuciones originales de área por variedad')
ax.tick_params(axis='x',labelrotation=35);fig.tight_layout()
fig.savefig(OUT/'NB17_Fig_01_perfiles_area_dry_bean.png',dpi=200);plt.show()
# Relación con confusiones (mismos registros del test NB12)
base=PRED[('dry_bean','R1')][['record_id','class_label','duplicate_group_id']].copy()
base=base.merge(dry[['record_id','Area','Class']],on='record_id',validate='one_to_one')
assert (base.class_label==base.Class).all()
for label in ('R1','R3-C','R4-C'):
    f=PRED[('dry_bean',label)]
    bp,_=probs(f)
    _,cls=probs(f)
    base['pred_'+label]=cls[bp.argmax(1)]
focus=base[base.class_label.isin(['SIRA','DERMASON'])].copy()
rows=[]
for cl in ['SIRA','DERMASON']:
    part=focus[focus.class_label==cl]
    for rep in ['R1','R3-C','R4-C']:
        wrong=part[part['pred_'+rep]!=cl]
        pair=part[part['pred_'+rep]==('DERMASON' if cl=='SIRA' else 'SIRA')]
        rows.append(dict(class_label=cl,representation=rep,n_test=len(part),n_wrong=len(wrong),
         n_confused_with_pair=len(pair),area_all_median=part.Area.median(),
         area_wrong_median=wrong.Area.median() if len(wrong) else np.nan,
         area_pair_confusion_median=pair.Area.median() if len(pair) else np.nan))
pair_df=pd.DataFrame(rows)
pair_df.to_csv(OUT/'NB17_02_SIRA_DERMASON_area_errores.csv',index=False)
display(pair_df.round(2))
fig,ax=plt.subplots(figsize=(9,5))
for label in ['SIRA','DERMASON']:
    dat=dry.loc[dry.Class==label,'Area']
    ax.hist(dat,bins=35,alpha=.45,density=True,label=f'{label} (n={len(dat)})')
ax.legend();ax.set_xlabel('Área (píxeles²)');ax.set_ylabel('Densidad')
ax.set_title('Dry Bean — superposición de perfiles de área (SIRA y DERMASON)')
fig.tight_layout();fig.savefig(OUT/'NB17_Fig_02_area_SIRA_DERMASON.png',dpi=200);plt.show()


## 4. Sensibilidad de ±0,005 / ±0,010 / ±0,020 y precisión inferencial
Bootstrap pareado **estratificado por clase y agrupado por `duplicate_group_id`** sobre predicciones congeladas. Se calculan IC del 90 % para TOST exploratorio y del 95 % para evidencia de diferencia. El margen ±0,01 **no está justificado agronómicamente**: este análisis muestra sensibilidad estadística, sin presentar una justificación práctica inexistente.

El MDD retrospectivo es una **aproximación normal de precisión**, no una potencia prospectiva confirmatoria ni una prueba de equivalencia.

In [ ]:
PAIRS={'dry_bean':[('R1','R3-C'),('R1','R4-C')],
       'iniap_rice':[('R1','R3-NR'),('R1','R4-NR')]}
def cluster_boot_delta(df,ra,rb,n_boot=N_BOOT,seed=SEED):
    yy=df.class_label.to_numpy(); aa=df[ra].to_numpy(); bb=df[rb].to_numpy()
    unique=sorted(df.class_label.unique())
    samples={}
    for cl,sub in df.groupby('class_label',sort=True):
        samples[cl]=[g.index.to_numpy() for _,g in sub.groupby('duplicate_group_id',sort=True)]
    rng=np.random.default_rng(seed); result=np.empty(n_boot)
    for i in range(n_boot):
        idx=[]
        for cl in unique:
            units=samples[cl]
            choices=rng.integers(0,len(units),len(units))
            idx.extend(np.concatenate([units[k] for k in choices]))
        ix=np.asarray(idx,dtype=int)
        result[i]=f1_score(yy[ix],aa[ix],labels=unique,average='macro')-f1_score(yy[ix],bb[ix],labels=unique,average='macro')
    return result
bs_rows=[];sensitivity=[]
for d,pairs in PAIRS.items():
    for a,b in pairs:
        fa=PRED[(d,a)];fb=PRED[(d,b)]
        PA,CL=probs(fa);PB,CLB=probs(fb)
        assert np.array_equal(CL,CLB)
        joint=fa[['record_id','class_label','duplicate_group_id']].copy()
        joint['a']=CL[PA.argmax(1)];joint['b']=CLB[PB.argmax(1)]
        samples=cluster_boot_delta(joint,'a','b',n_boot=N_BOOT,seed=SEED+len(bs_rows))
        point=f1_score(joint.class_label,joint.a,average='macro')-f1_score(joint.class_label,joint.b,average='macro')
        lo90,hi90=np.quantile(samples,[.05,.95]);lo95,hi95=np.quantile(samples,[.025,.975])
        sd=float(np.std(samples,ddof=1))
        mdd80=(norm.ppf(.975)+norm.ppf(.80))*sd
        min_margin=max(abs(lo90),abs(hi90))
        bs_rows.append(dict(dataset=d,contrast=f'{a} minus {b}',delta=point,ci90_lower=lo90,ci90_upper=hi90,
          ci95_lower=lo95,ci95_upper=hi95,bootstrap_sd=sd,normal_approx_MDD80_alpha05=mdd80,
          smallest_symmetric_margin_covering_ci90=min_margin,n_groups=joint.duplicate_group_id.nunique(),n_boot=N_BOOT))
        for margin in MARGINS:
            sensitivity.append(dict(dataset=d,contrast=f'{a} minus {b}',equivalence_margin=margin,
              ci90_lower=lo90,ci90_upper=hi90,equivalence_supported_exploratory=bool(lo90>-margin and hi90<margin),
              qualification='Retrospective test-set sensitivity only; not agronomic/operational margin justification'))
bs_df=pd.DataFrame(bs_rows);margin_df=pd.DataFrame(sensitivity)
bs_df.to_csv(OUT/'NB17_03_bootstrap_pareado_precision_y_MDD.csv',index=False)
margin_df.to_csv(OUT/'NB17_04_sensibilidad_margenes_equivalencia.csv',index=False)
display(bs_df.round(6));display(margin_df)
# Control: IC para los mismos contrastes del NB16 debe coincidir aproximadamente (semilla bootstrap distinta)
for _,z in ctrl.iterrows():
    rows=bs_df[(bs_df.dataset==z.dataset)&(bs_df.contrast==z.contrast.replace(' − ',' minus '))]
    if len(rows)==1:
        assert abs(rows.iloc[0].delta-z.macro_f1_delta)<1e-9
print('Equivalencia exploratoria: no convertir en conclusión agronómica hasta justificar márgenes.')


## 5. Calibración verdaderamente independiente y LAC / APS / RAPS / Mondrian
Para cada modelo campeón se reutiliza la partición **calibration** congelada, subdividida en tres partes disjuntas por grupos de duplicados:
- **40 %:** ajustar temperatura.
- **30 %:** umbral de clasificación selectiva (no se utiliza para conformal).
- **30 %:** estimar cuantiles de métodos conformales.

Se usa el mismo test congelado para **comparar**, nunca para elegir el método. RAPS fija `lambda = 0.01` y `k_reg = 1` antes de evaluar. Mondrian LAC tiene umbrales **por etiqueta candidata** basados solo en la calibración conformal. APS y RAPS aplican aleatorización uniforme reproducible a nivel de grupo. La interpretación formal de cobertura requiere intercambiabilidad de las unidades (o grupos).

In [ ]:
def cal_partition(data):
    groups=data[['duplicate_group_id','class_label']].drop_duplicates('duplicate_group_id').copy()
    if data.groupby('duplicate_group_id').class_label.nunique().gt(1).any():
        raise AssertionError('Grupos con distintas clases')
    ids=groups.duplicate_group_id.to_numpy();yy=groups.class_label.to_numpy()
    i1,rest=train_test_split(np.arange(len(groups)),train_size=.4,random_state=SEED,stratify=yy)
    i2,i3=train_test_split(rest,train_size=.5,random_state=SEED+1,stratify=yy[rest])
    names={'temperature':set(ids[i1]),'selective':set(ids[i2]),'conformal':set(ids[i3])}
    assert len(set.union(*names.values()))==len(ids)
    return {name:np.where(data.duplicate_group_id.isin(group_ids).to_numpy())[0] for name,group_ids in names.items()}
def temperature_scale(P,T):
    return softmax(np.log(np.clip(P,1e-12,1.0))/float(T),axis=1)
def fit_temperature(P,y,cl):
    fn=lambda logT:log_loss(y,temperature_scale(P,np.exp(logT)),labels=cl)
    res=minimize_scalar(fn,bounds=(-3,3),method='bounded')
    if not res.success:raise RuntimeError('Optimización de temperatura falló')
    return float(np.exp(res.x))
def higher_quantile(vals,alpha):
    n=len(vals);k=int(np.ceil((n+1)*(1-alpha)))
    return float(np.sort(vals)[k-1]) if k<=n else float('inf')
def group_random(group_ids,seed):
    uniques=sorted(set(map(str,group_ids)))
    rng=np.random.default_rng(seed)
    lookup=dict(zip(uniques,rng.random(len(uniques))))
    return np.asarray([lookup[str(x)] for x in group_ids],float)
def score_matrix(P,mode,U):
    P=np.asarray(P,float);n,k=P.shape
    if mode=='LAC' or mode=='Mondrian_LAC':return 1-P
    result=np.zeros((n,k),dtype=float)
    for i in range(n):
        rank=np.argsort(-P[i],kind='stable');cumsum=np.cumsum(P[i,rank]);before=cumsum-P[i,rank]
        s=before+U[i]*P[i,rank]
        if mode=='RAPS':s=s+RAPS_LAMBDA*np.maximum(np.arange(1,k+1)-RAPS_KREG,0)
        result[i,rank]=s
    return result

def conformal_for_dataset(dataset,rep):
    cal=load_pred(dataset,rep,'calibration');test=PRED[(dataset,rep)]
    C,cl=probs(cal);T,clt=probs(test)
    assert np.array_equal(cl,clt)
    split=cal_partition(cal); yy=cal.class_label.to_numpy(); yt=test.class_label.to_numpy()
    temp=fit_temperature(C[split['temperature']],yy[split['temperature']],cl)
    PC=temperature_scale(C,temp);PT=temperature_scale(T,temp)
    true_idx={lab:i for i,lab in enumerate(cl)}
    cal_idx=split['conformal'];group_cal=cal.iloc[cal_idx].sort_values('record_id').drop_duplicates('duplicate_group_id')
    idx_cal=group_cal.index.to_numpy()
    # Poblaciones seleccionadas a nivel de grupo para inferencia
    group_test=test.sort_values('record_id').drop_duplicates('duplicate_group_id')
    idx_test=group_test.index.to_numpy()
    uc=group_random(cal.duplicate_group_id.to_numpy(),seed=SEED+110)
    ut=group_random(test.duplicate_group_id.to_numpy(),seed=SEED+120)
    yycal=yy[idx_cal];true_pos_cal=np.array([true_idx[z] for z in yycal],int)
    true_pos_test=np.array([true_idx[z] for z in yt],int)
    glob=[];percl=[];quant=[]
    for method in ('APS_randomized','LAC','RAPS_fixed','Mondrian_LAC'):
        score_type={'APS_randomized':'APS','LAC':'LAC','RAPS_fixed':'RAPS','Mondrian_LAC':'Mondrian_LAC'}[method]
        sc=score_matrix(PC[idx_cal],score_type,uc[idx_cal]);st=score_matrix(PT,score_type,ut)
        true_scores=sc[np.arange(len(idx_cal)),true_pos_cal]
        for alpha in ALPHAS:
            if method!='Mondrian_LAC':
                q=higher_quantile(true_scores,alpha)
                sets=st<=q
                quant.append(dict(dataset=dataset,method=method,alpha=alpha,class_label='GLOBAL',q=q,n_cal=len(true_scores)))
            else:
                qs=np.empty(len(cl),float)
                for j,label in enumerate(cl):
                    scores_class=true_scores[yycal==label]
                    if len(scores_class)==0:raise ValueError('Mondrian sin calibración para '+label)
                    qs[j]=higher_quantile(scores_class,alpha)
                    quant.append(dict(dataset=dataset,method=method,alpha=alpha,class_label=label,q=qs[j],n_cal=len(scores_class)))
                sets=st<=qs[None,:]
            hits=sets[np.arange(len(yt)),true_pos_test];empty=~sets.any(axis=1)
            row_size=sets.sum(axis=1)
            glob.append(dict(dataset=dataset,representation=rep,method=method,nominal_coverage=1-alpha,
                n_calibration_groups=len(idx_cal),n_test_rows=len(yt),n_test_groups=len(idx_test),temperature=temp,
                row_coverage=hits.mean(),group_coverage=hits[idx_test].mean(),row_empty_fraction=empty.mean(),
                group_empty_fraction=empty[idx_test].mean(),mean_set_size=row_size.mean(),
                pct_multi_label=(row_size>1).mean()))
            for clabel in cl:
                mask=yt==clabel
                percl.append(dict(dataset=dataset,method=method,nominal_coverage=1-alpha,
                    class_label=clabel,n=int(mask.sum()),coverage=hits[mask].mean(),
                    empty_fraction=empty[mask].mean(),mean_set_size=row_size[mask].mean()))
    split_summary=dict(dataset=dataset,representation=rep,temperature_fit_n=len(split['temperature']),
        selective_threshold_n=len(split['selective']),conformal_n=len(split['conformal']),temperature=temp,
        calibration_groups=cal.duplicate_group_id.nunique(),test_groups=test.duplicate_group_id.nunique())
    return glob,percl,quant,split_summary
GLOBAL=[];CLASS=[];QUANT=[];SPLITS=[]
for dataset,champ in [('dry_bean','R1'),('iniap_rice','R4-NR')]:
    g,c,q,s=conformal_for_dataset(dataset,champ)
    GLOBAL.extend(g);CLASS.extend(c);QUANT.extend(q);SPLITS.append(s)
conformal_global=pd.DataFrame(GLOBAL)
conformal_by_class=pd.DataFrame(CLASS)
conformal_q=pd.DataFrame(QUANT)
split_out=pd.DataFrame(SPLITS)
conformal_global.to_csv(OUT/'NB17_05_conformal_global_LAC_APS_RAPS_Mondrian.csv',index=False)
conformal_by_class.to_csv(OUT/'NB17_06_conformal_cobertura_por_clase.csv',index=False)
conformal_q.to_csv(OUT/'NB17_07_cuantiles_mondrian_global.csv',index=False)
split_out.to_csv(OUT/'NB17_08_particiones_independientes_calibracion.csv',index=False)
display(split_out)
display(conformal_global[conformal_global.nominal_coverage==.9].round(4))


## 6. Figuras comparativas y recomendaciones condicionadas
Una cobertura global cercana al 90 % no implica cobertura del 90 % para cada clase. Las diferencias descriptivas de Mondrian se evalúan sobre el mismo test y no implican una garantía de validez para nuevas campañas ni diferentes dispositivos.

In [ ]:
sub=conformal_global[np.isclose(conformal_global.nominal_coverage,.90)]
for dataset in ('dry_bean','iniap_rice'):
    a=sub[sub.dataset==dataset].copy()
    fig,ax=plt.subplots(figsize=(10,5))
    ax.bar(a.method,a.row_coverage)
    ax.axhline(.90,linestyle='--',linewidth=1,label='Objetivo nominal 90 %')
    ax.set_ylim(0,1.03);ax.set_ylabel('Cobertura global en test')
    ax.set_title(f'{dataset} — métodos conformales con calibración independiente')
    ax.tick_params(axis='x',labelrotation=25);ax.legend();fig.tight_layout()
    fig.savefig(OUT/f'NB17_Fig_03_cobertura_global_{dataset}.png',dpi=200);plt.show()
    a=conformal_by_class[(conformal_by_class.dataset==dataset)&np.isclose(conformal_by_class.nominal_coverage,.9)]
    pv=a.pivot(index='class_label',columns='method',values='coverage')
    fig,ax=plt.subplots(figsize=(11,5))
    X=np.arange(len(pv)); width=.18
    for j,m in enumerate(pv.columns):ax.bar(X+(j-(len(pv.columns)-1)/2)*width,pv[m],width,label=m)
    ax.axhline(.9,linestyle='--',linewidth=1)
    ax.set_xticks(X,pv.index,rotation=25);ax.set_ylim(0,1.03);ax.legend(fontsize=8,ncol=2)
    ax.set_ylabel('Cobertura por clase en test')
    ax.set_title(f'{dataset} — desigualdad de cobertura entre clases (nominal 90 %)')
    fig.tight_layout();fig.savefig(OUT/f'NB17_Fig_04_cobertura_clases_{dataset}.png',dpi=200);plt.show()
# Nota científica; no seleccionar el mejor método ex post.
(OUT/'NB17_09_nota_metodologica.txt').write_text("\n".join([
    "ENSAYO ADICIONAL POR REVISORES — SIN CAMBIAR EL TEST CONGELADO.",
    "1) Area: evidencia descriptiva en pixeles cuadrados, sin atribucion causal.",
    "2) Margenes 0.005, 0.01 y 0.02: sensibilidad post hoc, no justificacion agronomica.",
    "3) El MDD80 es una aproximacion normal retrospectiva, no potencia prospectiva.",
    "4) LAC, APS, RAPS fijo y Mondrian utilizan calibracion independiente del ajuste de temperatura.",
    "5) No se seleccionan metodos observando el test.",
    "6) Validez grupal condicionada a intercambiabilidad entre grupos."
]) + "\n", encoding='utf-8')


## 7. Exportación automática a ZIP (el ÚNICO archivo que debes enviar)
Esta última celda crea una carpeta de resultados específica y un ZIP con metadatos e integridad SHA-256. Al terminar **se descarga automáticamente** en Colab y queda copia en Drive. No enviar el ZIP del NB16.

In [ ]:
META={'manuscript':'agriengineering-4664292','notebook':'NB17_Ensayos_complementarios_CPU',
      'run_utc':pd.Timestamp.now(tz='UTC').isoformat(),
      'source_nb12_sha256':hashlib.sha256(NB12_ZIP.read_bytes()).hexdigest(),
      'source_nb16_sha256':hashlib.sha256(NB16_ZIP.read_bytes()).hexdigest(),
      'frozen_dry_bean_sha256':sha, 'seed':SEED,'bootstraps':N_BOOT,
      'margins':list(MARGINS),'RAPS_lambda_fixed':RAPS_LAMBDA,'RAPS_kreg_fixed':RAPS_KREG,
      'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,
      'execution':'CPU','notes':'Post-review analysis. No model retraining or primary frozen split reassignment.'}
(OUT/'NB17_10_metadatos.json').write_text(json.dumps(META,ensure_ascii=False,indent=2),encoding='utf-8')
files_to_hash=sorted(p for p in OUT.iterdir() if p.is_file() and p.name.startswith('NB17_') and p.name!='NB17_11_SHA256.csv')
hashes=pd.DataFrame([dict(file=p.name,bytes=p.stat().st_size,sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in files_to_hash])
hashes.to_csv(OUT/'NB17_11_SHA256.csv',index=False)
all_files=sorted(p for p in OUT.iterdir() if p.is_file() and p.name.startswith('NB17_'))
archive_path=REVIEW/'NB17_ENSAYOS_COMPLEMENTARIOS_4664292.zip'
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in all_files:z.write(p,arcname='NB17_ENSAYOS_COMPLEMENTARIOS_4664292/'+p.name)
with zipfile.ZipFile(archive_path) as z:
    assert z.testzip() is None and len(z.namelist())==len(all_files)
print('✅ NB17 completado |',len(all_files),'archivos | ZIP:',archive_path)
print('✅ SHA256 ZIP:',hashlib.sha256(archive_path.read_bytes()).hexdigest())
print('📤 ENVÍAME ÚNICAMENTE:',archive_path.name)
from google.colab import files as colab_files
colab_files.download(str(archive_path))
